# Train, Tune, and Assess Split Sensitivity

This notebook repeats the complete 80/10/10 split, tuning, fitting, and assessment workflow for five independent data-split seeds. Every realization saves its own train, familiar-test, and leave-out-of-scene (LOS) CSVs; DT, RF, and XGBoost models; tuning studies; and metrics. The final tables compare balanced accuracy across realizations.

In [ ]:
from dataclasses import replace
import importlib
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

import lswt_cloud_masking.data_splitting as data_splitting
import lswt_cloud_masking.model_training as model_training
import lswt_cloud_masking.training_sensitivity as training_sensitivity

# Reload in dependency order so rerunning this notebook cannot retain stale dataclasses.
importlib.reload(data_splitting)
importlib.reload(model_training)
importlib.reload(training_sensitivity)

from lswt_cloud_masking.training_sensitivity import SensitivityConfig, run_sensitivity_pipeline

## 1. Configure the experiment

Keep `SMOKE_TEST = True` for a quick one-split pipeline check. Smoke runs are recomputed so you can change their trial counts freely. After the check succeeds, set it to `False` for the complete five-realization sensitivity run. Completed matching full runs are resumable, so an interrupted full experiment does not need to start from the beginning.

In [ ]:
config = SensitivityConfig.from_json(ROOT / "configs" / "training_sensitivity.example.json")

SMOKE_TEST = True
if SMOKE_TEST:
    overrides = dict(config.training_overrides or {})
    overrides.update({
        "tuning_cv_seeds": [42],
        "cv_splits": 2,
        "pixel_cv_splits": 2,
        "n_trials_dt": 1,
        "n_trials_rf": 1,
        "n_trials_xgb": 1,
        "top_candidates_per_run": 1,
        "xgb_max_estimators": 100,
        "xgb_early_stopping_rounds": 10,
    })
    config = replace(
        config,
        output_dir=f"{config.output_dir}_smoke",
        split_seeds=config.split_seeds[:1],
        training_overrides=overrides,
        # Smoke settings are commonly edited between checks; recompute them.
        resume_completed_runs=False,
    )

output_root = (ROOT / config.output_dir).resolve()
print("Split seeds:", config.split_seeds)
print("Datasets will be written to:", output_root / "datasets")
print("Models will be written to:", output_root / "models")
print("Training overrides:", config.training_overrides)
config

## 2. Generate every split and train every model

Within each realization, LOS scenes are absent from both training and familiar test, and all required lake/class/sensor/season coverage checks are applied. Test and LOS scores are calculated only after model selection.

In [ ]:
sensitivity_result = run_sensitivity_pipeline(config, project_root=ROOT)
sensitivity_result["paths"]

## 3. Compare results

In [ ]:
metric_columns = [
    "run_index", "split_seed", "model",
    "train_balanced_accuracy",
    "test_inscene_balanced_accuracy",
    "los_balanced_accuracy",
    "selection_stability_score",
    "model_path",
]
sensitivity_result["results"][metric_columns]

In [ ]:
sensitivity_result["summary"]

In [ ]:
# Shows how much the held-out scene set changes between realizations.
sensitivity_result["los_scene_overlap"]

Each realization is stored below the configured output root in `datasets/run_*` and `models/run_*`. `sensitivity_results.csv` contains one row per realization/model, while `sensitivity_summary_by_model.csv` reports the across-realization mean, sample standard deviation, minimum, and maximum of train, familiar-test, and LOS balanced accuracy. Do not select a final model merely because it has the highest LOS score; that would turn the LOS assessment into a model-selection set.